# HiNet Lite NPU v1 Colab v4 Metrics Review

This notebook loads the Colab v4 run artifacts for `lab3_hinet_lite_npu_v1_colab_v4.ipynb` and plots the metrics for Google Drive run `/runs/065325_2404`.

Primary artifact paths:

- `metrics.csv`: `/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/065325_2404/metrics.csv`
- `summary.json`: `/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/065325_2404/summary.json`
- `best_checkpoint`: `/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/065325_2404/checkpoints/best.pth`

The v4 run resumed from the v3 best checkpoint at epoch 39, so the loader below also handles replayed history and duplicate epoch numbers by adding a monotonic `metric_step` index.


In [ ]:
from pathlib import Path
import csv
import json
import math
from statistics import mean

import matplotlib.pyplot as plt

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception:
    pass

RUN_ROOT = Path('/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/065325_2404')
METRICS_CSV_PATH = RUN_ROOT / 'metrics.csv'
SUMMARY_JSON_PATH = RUN_ROOT / 'summary.json'
BEST_CHECKPOINT_PATH = RUN_ROOT / 'checkpoints' / 'best.pth'
V3_RESUME_BEST_VAL_PSNR = 24.233901977539062
V3_RESUME_BEST_EPOCH = 39
TARGET_PSNR = 25.0

print({
    'run_root': str(RUN_ROOT),
    'metrics_csv_path': str(METRICS_CSV_PATH),
    'summary_json_path': str(SUMMARY_JSON_PATH),
    'best_checkpoint_path': str(BEST_CHECKPOINT_PATH),
})


In [ ]:
if not METRICS_CSV_PATH.exists():
    raise FileNotFoundError(f'Missing metrics.csv: {METRICS_CSV_PATH}')

summary = {}
if SUMMARY_JSON_PATH.exists():
    summary = json.loads(SUMMARY_JSON_PATH.read_text(encoding='utf-8'))
else:
    print(f'Warning: missing summary.json: {SUMMARY_JSON_PATH}')


def as_float(value, default=math.nan):
    if value in (None, ''):
        return default
    return float(value)


def as_bool(value) -> bool:
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() in {'1', 'true', 'yes'}

rows = []
with METRICS_CSV_PATH.open(newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    previous_epoch = None
    segment_id = 0
    for step, row in enumerate(reader, start=1):
        epoch = int(row['epoch'])
        if previous_epoch is not None and epoch <= previous_epoch:
            segment_id += 1
        previous_epoch = epoch
        train_eval_psnr = as_float(row.get('train_eval_psnr'))
        val_psnr = as_float(row.get('val_psnr'))
        gap = as_float(row.get('generalization_gap'), train_eval_psnr - val_psnr)
        rows.append({
            'metric_step': step,
            'segment_id': segment_id,
            'epoch': epoch,
            'epoch_time_sec': as_float(row.get('epoch_time_sec')),
            'lr': as_float(row.get('lr')),
            'train_loss_residual': as_float(row.get('train_loss_residual')),
            'train_patch_psnr': as_float(row.get('train_patch_psnr')),
            'train_eval_psnr': train_eval_psnr,
            'val_psnr': val_psnr,
            'input_psnr': as_float(row.get('input_psnr')),
            'delta_psnr': as_float(row.get('delta_psnr')),
            'generalization_gap': gap,
            'is_best': as_bool(row.get('is_best')),
        })

if not rows:
    raise RuntimeError(f'No metric rows found in {METRICS_CSV_PATH}')

best_row = max(rows, key=lambda r: r['val_psnr'])
last_segment_id = max(r['segment_id'] for r in rows)
last_segment_rows = [r for r in rows if r['segment_id'] == last_segment_id]
resume_epoch_rows = [r for r in rows if r['epoch'] > V3_RESUME_BEST_EPOCH]
current_run_rows = last_segment_rows if last_segment_id > 0 else (resume_epoch_rows or rows)
current_best_row = max(current_run_rows, key=lambda r: r['val_psnr'])
latest_row = rows[-1]

segment_counts = {segment: sum(1 for r in rows if r['segment_id'] == segment) for segment in sorted({r['segment_id'] for r in rows})}
summary_best = summary.get('best_val_psnr')
summary_epoch = summary.get('best_epoch')

review = {
    'metric_rows': len(rows),
    'segment_counts': segment_counts,
    'summary_best_epoch': summary_epoch,
    'summary_best_val_psnr': summary_best,
    'metrics_best_epoch': best_row['epoch'],
    'metrics_best_step': best_row['metric_step'],
    'metrics_best_val_psnr': best_row['val_psnr'],
    'current_run_best_epoch': current_best_row['epoch'],
    'current_run_best_step': current_best_row['metric_step'],
    'current_run_best_val_psnr': current_best_row['val_psnr'],
    'latest_epoch': latest_row['epoch'],
    'latest_val_psnr': latest_row['val_psnr'],
    'gain_over_v3_best_db': current_best_row['val_psnr'] - V3_RESUME_BEST_VAL_PSNR,
    'gap_to_25db': TARGET_PSNR - current_best_row['val_psnr'],
    'best_delta_over_input_db': current_best_row['delta_psnr'],
    'best_generalization_gap_db': current_best_row['generalization_gap'],
    'best_checkpoint': str(BEST_CHECKPOINT_PATH),
    'stop_reason': summary.get('stop_reason'),
}
print(json.dumps(review, indent=2))


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 14), sharex=True)

for segment_id in sorted({r['segment_id'] for r in rows}):
    segment = [r for r in rows if r['segment_id'] == segment_id]
    label_suffix = 'current v4 segment' if segment is current_run_rows else f'segment {segment_id}'
    axes[0].plot([r['metric_step'] for r in segment], [r['train_eval_psnr'] for r in segment], linewidth=1.8, label=f'Train eval PSNR ({label_suffix})')
    axes[0].plot([r['metric_step'] for r in segment], [r['val_psnr'] for r in segment], linewidth=2.4, label=f'Val PSNR ({label_suffix})')

axes[0].axhline(TARGET_PSNR, color='black', linestyle='--', linewidth=1.2, label='25 dB target')
axes[0].axhline(V3_RESUME_BEST_VAL_PSNR, color='gray', linestyle=':', linewidth=1.2, label='v3 resume best')
axes[0].scatter([current_best_row['metric_step']], [current_best_row['val_psnr']], color='crimson', zorder=4, label=f"v4 best: epoch {current_best_row['epoch']} ({current_best_row['val_psnr']:.4f} dB)")
axes[0].set_title('HiNet Lite NPU v1 Colab v4 PSNR')
axes[0].set_ylabel('PSNR (dB)')
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=8, ncol=2)

axes[1].plot([r['metric_step'] for r in rows], [r['train_patch_psnr'] for r in rows], label='Train patch PSNR', linewidth=1.8)
axes[1].plot([r['metric_step'] for r in rows], [r['train_loss_residual'] for r in rows], label='Residual MSE loss', linewidth=1.8)
axes[1].set_title('Training Signal')
axes[1].set_ylabel('Value')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

axes[2].plot([r['metric_step'] for r in rows], [r['generalization_gap'] for r in rows], label='Train eval - val gap', linewidth=2)
axes[2].plot([r['metric_step'] for r in rows], [r['delta_psnr'] for r in rows], label='Val gain over LR input', linewidth=2)
axes[2].axvline(current_best_row['metric_step'], color='crimson', linestyle='--', linewidth=1.2, label='v4 best step')
axes[2].set_title('Generalization and Input Baseline Gain')
axes[2].set_xlabel('Metric row / replay-aware step')
axes[2].set_ylabel('dB')
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()


In [ ]:
current_tail = current_run_rows[-min(8, len(current_run_rows)):]
print('Recent current-run rows:')
for r in current_tail:
    print(
        f"step={r['metric_step']:03d} epoch={r['epoch']:03d} "
        f"lr={r['lr']:.6f} train_patch={r['train_patch_psnr']:.4f} "
        f"train_eval={r['train_eval_psnr']:.4f} val={r['val_psnr']:.4f} "
        f"gap={r['generalization_gap']:.4f} delta={r['delta_psnr']:.4f} "
        f"is_best={r['is_best']}"
    )

if len(current_run_rows) >= 2:
    after_best = [r for r in current_run_rows if r['metric_step'] > current_best_row['metric_step']]
    if after_best:
        print({
            'post_best_rows': len(after_best),
            'post_best_val_change_db': after_best[-1]['val_psnr'] - current_best_row['val_psnr'],
            'post_best_train_patch_change_db': after_best[-1]['train_patch_psnr'] - current_best_row['train_patch_psnr'],
            'mean_current_epoch_time_sec': mean(r['epoch_time_sec'] for r in current_run_rows if not math.isnan(r['epoch_time_sec'])),
        })


## v4 Performance Interpretation

The v4 run is a useful stage-2 improvement, but it is not yet enough for the `>25 dB` Lab 3 scoring threshold. It resumed from v3 best validation PSNR `24.2339 dB`, reached `24.3264 dB` at epoch `46`, then early-stopped at epoch `52` after six epochs without validation improvement. The best v4 checkpoint improves the LR input baseline by about `1.2034 dB`, but remains about `0.6736 dB` below `25 dB`.

The main signal is not under-training. After the best epoch, train patch PSNR keeps rising while train-eval and validation PSNR flatten or decline. At the v4 best point, train-eval PSNR is about `26.0215 dB` while validation PSNR is `24.3264 dB`, a gap of about `1.6951 dB`. That points to generalization/data-match limits more than raw capacity limits.

Recommended next improvement experiments, using Modal/Colab only:

- Keep v4 architecture fixed and run a short low-LR polish from the epoch-46 checkpoint: `lr=1e-5` to `2e-5`, patience `3`, no warmup, compare EMA vs raw. This tests whether v4 overshot with `6e-5`.
- Try the fresh-mode loss on the v4 resume path: residual Charbonnier/L1 or mixed residual MSE + L1 can preserve PSNR while reducing the sharp train/val divergence seen with pure residual MSE.
- Increase train-eval coverage for monitoring from `128` to a larger deterministic subset or all training pairs periodically. The existing train-eval subset is small enough that the reported gap may be noisy.
- Add a validation-like fixed-center or multi-crop monitor from training pairs. Current random 256 crops with rotations/flips can diverge from validation behavior, so a second monitor will separate augmentation benefit from validation mismatch.
- Do not continue the same v4 schedule beyond epoch 46 without changing LR/loss/regularization; the recorded decline is consistent and already triggered early stopping.

## Parameter Reduction Starting Points

Current v4 uses `10,453,443` parameters with channels `48`, encoder `(3, 3)`, bottleneck `6`, decoder `(3, 3)`, and mixed `3x3/5x5` kernels. Use same-slice, same-budget comparisons before ranking these variants.

| Variant | Estimated params | Reduction | Why test it |
| --- | ---: | ---: | --- |
| `c44`, same depth/kernels | `8,784,339` | `16.0%` | Lowest-risk width trim; good first compression probe. |
| `c40`, same depth/kernels | `7,260,323` | `30.5%` | Meaningful param cut while preserving topology and receptive field. |
| `c48`, bottleneck `4` | `7,945,923` | `24.0%` | Cuts the most expensive low-resolution blocks while keeping high-res stages. |
| `c40`, bottleneck `4` | `5,518,883` | `47.2%` | Practical second-stage compression if `c40` or bottleneck-4 alone holds PSNR. |
| `c48`, all `3x3` kernels | `5,808,579` | `44.4%` | Large cut, but higher PSNR risk because it removes the 5x5 receptive-field path. |

First compression ladder: run `c44 same`, `c40 same`, and `c48 bottleneck4` against the v4 training budget. If one stays within roughly `0.05-0.10 dB` of v4, use it as the new base and only then test compound reductions like `c40 bottleneck4`. Avoid testing all-`3x3` first unless latency or MXQ conversion forces removal of 5x5 kernels.
